In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from utils.feature_engineering import create_features

df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "patient_financial_risk.csv"
)

df = create_features(df)

print(df.shape)
df.head()

(10000, 23)


,patient_id,age,gender,bmi,smoker,diabetes,hypertension,heart_disease,hospital_visits,medication_count,...,hospital_debt_ratio,pharma_investment_score,esg_score,pandemic_risk_index,future_health_cost,risk_level,chronic_condition_count,claims_to_premium_ratio,health_risk_score,high_cost_patient
0,1,69,Female,25.3,0,0,1,1,1,3,...,0.29,19.69,84.55,0.511,27769.38,High,2,2.001611,6.639,1
1,2,32,Male,31.0,0,0,0,0,1,1,...,0.42,2.03,66.94,0.801,13585.13,Low,0,1.987047,2.070,0
2,3,89,Male,39.2,0,0,1,0,1,3,...,0.41,27.45,61.35,0.410,17540.43,Medium,1,1.424980,5.456,1
3,4,78,Female,25.2,0,1,0,1,4,5,...,0.58,95.81,67.46,0.275,40978.02,High,2,2.828050,8.316,1
4,5,38,Female,36.3,0,0,0,0,2,1,...,0.53,57.89,35.44,0.942,14812.69,Medium,0,1.979055,2.849,0


In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(
    columns=[
        "patient_id",
        "future_health_cost",
        "risk_level"
    ]
)

X = pd.get_dummies(X, drop_first=True)

y = df["future_health_cost"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.20
)

print(X_test.shape)

(2000, 20)


In [4]:
ensemble_model = joblib.load(
    PROJECT_ROOT /
    "models" /
    "trained" /
    "stacking_ensemble_cost_model.pkl"
)

ensemble_model

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('xgb', ...), ('lgbm', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Ridge()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[XGBRegressor(...ree=None, ...), LGBMRegressor...ndom_state=42)]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadded:: 1.0","ndarray[<U23](20,)","['age','bmi','smoker',...,'health_risk_score','high_cost_patient', 'gender_Male']"
final_estimator_ final_estimator_: estimatorThe regressor fit on the output of `estimators_` and responsible forfinal predictions.,Ridge,Ridge()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,20


In [6]:
from explainability.counterfactual import CounterfactualExplainer

In [7]:
explainer = CounterfactualExplainer(
    model=ensemble_model,
    feature_columns=list(X_test.columns)
)

In [8]:
patient = X_test.iloc[[0]]

patient

,age,bmi,smoker,diabetes,hypertension,heart_disease,hospital_visits,medication_count,annual_claim_amount,premium_paid,hospital_rating,hospital_debt_ratio,pharma_investment_score,esg_score,pandemic_risk_index,chronic_condition_count,claims_to_premium_ratio,health_risk_score,high_cost_patient,gender_Male
6252,87,21.2,1,1,0,1,8,2,33404.16,12415,2,0.28,99.84,61.3,0.416,2,2.690629,10.376,1,False


In [9]:
result = explainer.explain(
    patient_row=patient,
    top_n=3
)

In [10]:
print(result["executive_summary"])

Baseline predicted healthcare cost is $44,127.12. The strongest actionable counterfactual is: Improve claims-to-premium ratio by 0.10. This changes claims_to_premium_ratio from 2.691 to 2.591 and may reduce cost to $43,429.63, saving approximately $697.49 (1.58%).


In [11]:
recommendations = pd.DataFrame(
    result["top_recommendations"]
)

recommendations

,feature,recommendation,old_value,new_value,baseline_cost,counterfactual_cost,estimated_savings,estimated_savings_pct,confidence,difficulty
0,claims_to_premium_ratio,Improve claims-to-premium ratio by 0.10,2.691,2.591,44127.12,43429.63,697.49,1.58,0.7,Medium
1,health_risk_score,Improve overall health risk score by 0.50,10.376,9.876,44127.12,43965.15,161.98,0.37,0.7,Medium
2,pandemic_risk_index,Reduce pandemic exposure risk index by 0.10,0.416,0.316,44127.12,44266.58,-139.45,-0.32,0.5,Low


In [12]:
for i in range(5):

    patient = X_test.iloc[[i]]

    result = explainer.explain(patient)

    print("=" * 80)
    print(f"Patient {i+1}")
    print(result["executive_summary"])

Patient 1
Baseline predicted healthcare cost is $44,127.12. The strongest actionable counterfactual is: Improve claims-to-premium ratio by 0.10. This changes claims_to_premium_ratio from 2.691 to 2.591 and may reduce cost to $43,429.63, saving approximately $697.49 (1.58%).
Patient 2
Baseline predicted healthcare cost is $17,952.49. The strongest actionable counterfactual is: Reduce medication burden by 1 where clinically appropriate. This changes medication_count from 3.0 to 2.0 and may reduce cost to $17,905.07, saving approximately $47.42 (0.26%).
Patient 3
Baseline predicted healthcare cost is $14,966.60. The strongest actionable counterfactual is: Reduce pandemic exposure risk index by 0.10. This changes pandemic_risk_index from 0.835 to 0.735 and may reduce cost to $14,723.95, saving approximately $242.65 (1.62%).
Patient 4
Baseline predicted healthcare cost is $13,579.32. The strongest actionable counterfactual is: Reduce pandemic exposure risk index by 0.10. This changes pandem